In [ ]:
df=master

In [ ]:
def generate_dataframe_code_agent(task: str, model: str = "openai:o4-mini") -> str:
    """
    Generates Python pandas code for a specific dataframe manipulation task.

    Args:
        task (str): The dataframe task to generate code for.
        model (str): Language model to use.

    Returns:
        str: Python code as a string.
    """
    user_prompt = f"""
You are a Python data-processing assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.
Assume `df` already exists as a pandas DataFrame.

Rules:
- Produce executable pandas code.
- Prefer clear variable names.
- Add short comments for each section.
- Print useful diagnostics when appropriate.
- Do not redefine df.
- Do not import unnecessary packages.

Task: Write pandas code to convert these one-hot columns into a single multiclass target y:
- study_group_healthy
- study_group_pre_diabetes_lifestyle_controlled
- study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled
- study_group_insulin_dependent

Requirements:
- Cast the columns to int
- Verify each row has exactly one active class
- Print invalid row count where sum != 1
- Create:
    class_names = ["healthy", "prediabetes", "oral_medicated", "insulin_dependent"]
- Create y as the class index using idxmax or numpy
- Print y value counts
- Print df shape and columns before split
- Return only Python code
"""
    messages = [{"role": "user", "content": user_prompt}]

    response = CLIENT.chat.completions.create(
        model=model,
        messages=messages,
        temperature=1
    )

    code_str = response.choices[0].message.content.strip()
    return code_str

In [ ]:
def generate_target_code_agent(model: str = "openai:o4-mini") -> str:
    """
    Generates Python code for building y from the 4 one-hot target columns.
    """
    user_prompt = """
You are a Python machine learning assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume:
- `df` already exists as a pandas DataFrame

Your output must look very similar in style and structure to this format:
- section header comments
- short Goal comment
- exact variable name `target_cols`
- exact variable name `class_names`
- print diagnostics

You MUST generate code equivalent to this step:

#--------------------------------------
# 1.) Build y from one-hot cols
#----------------------------------------
#Goal: Turn 4 one-hot target columns into a single multi-class label

target_cols = [
    "study_group_healthy",
    "study_group_pre_diabetes_lifestyle_controlled",
    "study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled",
    "study_group_insulin_dependent"
]

df[target_cols] = df[target_cols].astype(int)
print(df[target_cols].sum(axis=1).value_counts())

class_names = ["healthy", "prediabetes", "oral_medicated", "insulin_dependent"]

row_sums = df[target_cols].sum(axis=1)
print(row_sums.value_counts())
print("Bad rows (sum!=1):", (row_sums != 1).sum())

##########################

print("just before split section df columns (n):", df.shape[1])
print(df.columns.tolist())

Rules:
- Use EXACTLY the 4 target column names shown above
- Use EXACTLY the 4 class names shown above
- Keep the same notebook-style formatting
- Keep the same comment style
- Do not add extra steps
- Return only Python code
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def splitting_code_agent(model: str = "openai:o4-mini") -> str:
    """
   Build feature matrices and class labels using the datasets pre made split columns
    """
    user_prompt = """
You are a Python machine learning assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume:
- `df` already exists as a pandas DataFrame

Your output must look very similar in style and structure to this format:
- section header comments
- short Goal comment
- exact variable name `target_cols`
- exact variable name `class_names`
- print diagnostics

You MUST generate code equivalent to this step:

#---------------------------
# 2.) Train/Val/Test Split use provided splits
#---------------------------
#Goal: Build feature matrices and class labels using the datasets pre made split columns

train_df = df[df["split_train"] == 1].copy()
val_df   = df[df["split_val"] == 1].copy()
test_df  = df[df["split_test"] == 1].copy()

#removing the target one-hot columns
drop_cols = target_cols + ["participant_id", "recommended_split", "split_train", "split_val", "split_test", "target"]

#c is each column name, only include in the dataframe
X_train = train_df.drop(columns=[c for c in drop_cols if c in train_df.columns]).copy()
y_train = train_df[target_cols].values.argmax(axis=1)

# Make groups aligned with X_train rows
groups_train = train_df.loc[X_train.index, "participant_id"].to_numpy()

X_val = val_df.drop(columns=[c for c in drop_cols if c in val_df.columns]).copy()
y_val = val_df[target_cols].values.argmax(axis=1)

X_test = test_df.drop(columns=[c for c in drop_cols if c in test_df.columns]).copy()
y_test = test_df[target_cols].values.argmax(axis=1)

##############################

Rules:
- Use EXACTLY the 3 split column names shown above
- Keep the same notebook-style formatting
- Keep the same comment style
- Do not add extra steps
- Return only Python code
- Remove anything regarding 'targets' from X_train!
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def generate_preprocessor_code_agent(model: str = "openai:o4-mini") -> str:
    """
    Generates Python code for preprocessing tabular data
    using numeric/categorical pipelines and a ColumnTransformer.
    """
    user_prompt = """
You are a Python machine learning coding assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume the following variables already exist:
- X_train
- X_val
- X_test
- y_train
- train_df
- target_cols
- np
- Pipeline
- SimpleImputer
- StandardScaler
- OneHotEncoder
- ColumnTransformer
- compute_class_weight

Task:
Write code for this pipeline step in a clean notebook style.

Required behavior:
1. Identify numeric and categorical features from X_train
   - numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
   - categorical_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

2. Print diagnostics:
   - X_train columns
   - count of numeric features
   - first 20 numeric features
   - count of categorical features
   - first 20 categorical features

3. Build:
   - numeric_pipe with median imputation + StandardScaler
   - categorical_pipe with most_frequent imputation + OneHotEncoder(handle_unknown="ignore")

4. Build a ColumnTransformer named preprocessor with:
   - ("num", numeric_pipe, numeric_features)
   - ("cat", categorical_pipe, categorical_features)
   - remainder="drop"

5. Fit on X_train and transform X_train, X_val, X_test into:
   - X_train_p
   - X_val_p
   - X_test_p

6. If outputs are sparse, convert them to dense arrays using:
   - .toarray() if available

7. Compute balanced class weights from y_train using compute_class_weight
   - store in classes
   - store raw weights in weights
   - create final_cw = dict(zip(classes, weights))
   - cap each weight at 6.0
   - ensure values are floats

8. Print diagnostics:
   - np.bincount(y_train, minlength=4)
   - train_df[target_cols].sum().to_dict()

Style requirements:
- Use section comments in this exact style:
  # ----------------------------------
  # 3.) Preprocessor
  # ------------------------------------
- Include a short Goal comment
- Keep the code compact but readable
- Return only Python code
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def generate_best_mlp_agent(task: str, model: str = "openai:o4-mini") -> str:
    user_prompt = f"""
You are an expert tabular deep learning engineer.

Your job is to design the best-performing MLP pipeline for this task:

{task}

Context:
- This is a multiclass classification problem with 4 classes.
- The data has already been preprocessed.
- Data is already split into train/validation/test.
- The solution must be practical, reproducible, and aimed at maximizing validation performance.
- Use ONLY these existing variables:
  - X_train_p
  - X_val_p
  - X_test_p
  - y_train
  - y_val
  - y_test
  - np
  - tf
- Class weights may already exist as `final_cw`.
- Do NOT redefine X_train, X_val, X_test.

Optimization objective:
- Maximize validation macro F1 score.
- Secondary goals: stable training, reasonable training time, and good minority-class performance.

You must produce Python code that defines BOTH:
1. A reusable function exactly named:
   build_mlp
2. A Python dictionary exactly named:
   best_params

Requirements:
1. Define a reusable function exactly named build_mlp with EXACTLY this signature:

def build_mlp(
    n_in,
    n_out=4,
    hidden_units=[256, 128, 64],
    dropout_rates=[0.3, 0.3, 0.2],
    activation="relu",
    normalization="batch",
    lr=1e-3,
    optimizer_name="adam",
    loss_name="sparse_categorical_crossentropy"
):

2. Do not rename these arguments.
3. Later pipeline steps will call build_mlp using n_in and n_out.
4. Use n_in to set the input shape.
5. Use n_out to set the output layer size.
6. The final output layer MUST be:
   tf.keras.layers.Dense(n_out, activation="softmax")
7. The model MUST compile with:
   - loss="sparse_categorical_crossentropy"
   - metrics=["accuracy"]
8. Do NOT use tf.keras.metrics.Precision, tf.keras.metrics.Recall, tf.keras.metrics.AUC, TopK metrics, or confusion-matrix-based metrics.
9. Do NOT use categorical_crossentropy, binary_crossentropy, or one-hot labels.
10. Store the winning hyperparameters in best_params using these exact keys:
   - hidden_units
   - dropout_rates
   - activation
   - normalization
   - lr
   - optimizer_name
   - loss_name
   - batch_size

11. Use the existing preprocessed matrices only.
12. Do not create a preprocessing pipeline.
13. Do not use OneHotEncoder.
14. Do not use ColumnTransformer.
15. Use early stopping.
16. Track results for all tried configurations.
17. Choose the best model using validation macro F1 computed AFTER prediction on y_val:
   - get validation probabilities from model.predict(X_val_p)
   - convert to predicted class ids with argmax(axis=1)
   - compute macro F1 from y_val and predicted labels
18. Evaluate the selected best model on validation and test sets.
19. Print a concise summary of why the selected configuration won.
20. build_mlp and best_params must remain available in scope.

Rules:
- Return ONLY valid Python code.
- Do not include markdown fences.
- Assume standard ML Python libraries are available.
- Prefer robust, realistic defaults over overly experimental choices.
- Include comments.
- If loss_name is "sparse_categorical_crossentropy", do not pass label_smoothing.
- Use plain "sparse_categorical_crossentropy" or tf.keras.losses.SparseCategoricalCrossentropy() with no extra arguments.
- Make the code deterministic where practical by setting random seeds.
"""
    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def generate_groupkfold_code_agent(model: str = "openai:o4-mini") -> str:
    """
    Generates Python code for GroupKFold cross-validation
    with preprocessing inside each fold and MLP training.
    """
    user_prompt = """
You are a Python machine learning coding assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume these variables already exist:
- X_train
- y_train
- groups_train
- preprocessor
- build_mlp
- np
- GroupKFold
- compute_class_weight
- EarlyStopping
- accuracy_score
- f1_score

Task:
Write code for this exact notebook section:

# --------------------------
# 5.) GroupKFold CV
# --------------------------
# Goal: Estimate performance robustly while preventing leakage between related samples

Requirements:
1. Create:
   - gkf = GroupKFold(n_splits=5)
   - fold_acc, fold_f1 = [], []

2. Loop over:
   - for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups=groups_train), start=1):

3. Inside each fold:
   - create X_tr, X_va using X_train.iloc[tr_idx] and X_train.iloc[va_idx]
   - create y_tr, y_va using y_train indexing (NO ".iloc" is need for y_train)
   - fit preprocessor on X_tr only
   - transform X_tr and X_va
   - densify sparse matrices with toarray() if available

4. Compute balanced class weights from y_tr
   - create classes = np.unique(y_tr)
   - compute weights using:
     compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
   - create final_cw from dict(zip(classes, weights))
   - cap weights at 4.5
   - print class_weight
   
5. Build and train the MLP:
   - build the model exactly like this pattern:

     mlp = build_mlp(
         n_in=X_tr_p.shape[1],
         n_out=4,
         hidden_units=best_params.get("hidden_units", [256, 128, 64]),
         dropout_rates=best_params.get("dropout_rates", [0.3, 0.3, 0.2]),
         activation=best_params.get("activation", "relu"),
         normalization=best_params.get("normalization", "batch"),
         lr=best_params.get("lr", 1e-3),
         optimizer_name=best_params.get("optimizer_name", "adam"),
         loss_name=best_params.get("loss_name", "sparse_categorical_crossentropy"),
     )

   - create:
     early_stop = EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)

   - create:
     reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

   - set:
     batch_size = best_params.get("batch_size", 32)

   - fit with:
       class_weight=final_cw
       validation_data=(X_va_p, y_va)
       epochs=best_params.get("epochs", 200)
       batch_size=batch_size
       callbacks=[early_stop, reduce_lr]
       verbose=0     
    
6. Predict probabilities on X_va_p

7. For thresholds [0.20, 0.25, 0.30, 0.35]:
   - create y_pred from argmax
   - override class to 3 where probs[:, 3] > t
   - compute insulin recall using recall_score(y_va, y_pred, labels=[3], average=None)[0]
   - print threshold and insulin recall

8. Then choose one fixed threshold t = 0.30 to score fold metrics:
   - y_pred = np.argmax(probs, axis=1)
   - y_pred[probs[:, 3] > t] = 3
   - append accuracy_score to fold_acc
   - append weighted f1_score to fold_f1

9. Then print insulin precision, recall, and f1 for thresholds [0.20, 0.25, 0.30, 0.35]
   - import precision_score, recall_score, f1_score locally before this block
   - compute and print insulin P/R/F1

10. After all folds, print:
   - CV mean and std for accuracy
   - CV mean and std for weighted F1

11. Also create feedback variables:
    - cv_summary: a dictionary containing the computed mean/std metrics
    - cv_feedback: a dictionary summarizing the main weakness and priority
    - round_score = cv_summary["mean_f1"]

Example structure only (do not hardcode values):
   -    cv_summary = {
        "mean_acc": 0.981,
        "std_acc": 0.015,
        "mean_f1": 0.982,
        "std_f1": 0.014,
        "mean_insulin_recall_030": 0.74,
        "mean_insulin_precision_030": 0.89,
        "mean_insulin_f1_030": 0.80,}

    - cv_feedback = {
    "main_issue": "low insulin recall" if mean_insulin_recall_030 < threshold else "balanced",
    "priority": "improve class 3 recall without collapsing precision",
    "fold_f1": [...]
}

   -round_score = cv_summary["mean_f1"]
   
Style requirements:
- Keep formatting similar to a clean notebook
- Include short comments
- Return only Python code
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def revise_mlp_agent(task: str, prior_code: str, cv_summary: dict, cv_feedback: dict, model: str = "openai:o4-mini") -> str:
    user_prompt = f"""
You are a Python machine learning coding assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Task:
Improve the current MLP for a 4-class tabular classification problem.

Current objective:
{task}

Current MLP code:
{prior_code}

Cross-validation summary:
{cv_summary}

Cross-validation feedback:
{cv_feedback}

Requirements:
1. Return only Python code.
2. Define:
   - build_mlp
   - best_params
3. Keep the same function signature expected by the notebook.
4. Try to improve weighted F1, but specifically address:
   - insulin_dependent recall
   - confusion with oral_medicated
5. Prefer stable, realistic changes:
   - hidden_units
   - dropout_rates
   - lr
   - batch_size
   - normalization
   - optimizer_name
   - loss_name
6. Do not invent new external dependencies.

Compatibility requirement:
- Do not use label_smoothing with SparseCategoricalCrossentropy.
- Use plain "sparse_categorical_crossentropy" or tf.keras.losses.SparseCategoricalCrossentropy() with no extra arguments.

"""
    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def run_mlp_feedback_loop(scope, initial_mlp_code, n_rounds=3, task="Improve the MLP"):
    best_code = initial_mlp_code
    best_score = -1
    best_cv_summary = None
    best_cv_feedback = None
    round_history = []

    mlp_code = initial_mlp_code

    for round_idx in range(1, n_rounds + 1):
        print(f"\n=== Feedback round {round_idx} ===")

        
        exec(mlp_code, scope)

        
        gkf_code = generate_groupkfold_code_agent()
        exec(gkf_code, scope)

        cv_summary = scope["cv_summary"]
        cv_feedback = scope["cv_feedback"]
        round_score = scope["round_score"]

        print("cv_summary:", cv_summary)
        print("cv_feedback:", cv_feedback)
        print("round_score:", round_score)

        round_history.append({
            "round": round_idx,
            "score": round_score,
            "cv_summary": cv_summary,
            "cv_feedback": cv_feedback,
            "mlp_code": mlp_code,
        })

        if round_score > best_score:
            best_score = round_score
            best_code = mlp_code
            best_cv_summary = cv_summary
            best_cv_feedback = cv_feedback

        if round_idx < n_rounds:
            mlp_code = revise_mlp_agent(
                task=task,
                prior_code=mlp_code,
                cv_summary=cv_summary,
                cv_feedback=cv_feedback,
            )

    return {
        "best_code": best_code,
        "best_score": best_score,
        "best_cv_summary": best_cv_summary,
        "best_cv_feedback": best_cv_feedback,
        "round_history": round_history,
    }

In [ ]:
def generate_final_training_code_agent(model: str = "openai:o4-mini") -> str:
    """
    Generates Python code for final model training on TRAIN,
    early stopping on VAL, evaluation on TEST, and artifact saving.
    """
    user_prompt = """
You are a Python machine learning coding assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume these variables already exist:
- preprocessor
- X_train
- X_val
- X_test
- y_train
- y_val
- y_test
- build_mlp
- compute_class_weight
- EarlyStopping
- ReduceLROnPlateau
- np
- class_names

Task:
Write code for this notebook section:

# ------------------------
# 6.) Final Model Training
# -------------------------
# Goal: Train one final model on TRAIN, tune/early-stop on VAL, then evaluate on TEST

Requirements:
1. Fit the preprocessor on X_train only, then transform:
   - X_train_p
   - X_val_p
   - X_test_p

2. Convert sparse outputs to dense arrays using .toarray() if available.

3. Compute balanced class weights from y_train:
   - classes = np.unique(y_train)
   - weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
   - final_cw = dict(zip(classes, weights))
   - cap each class weight at 6.0
   - keep values as floats

4. Build final model:
   - final_mlp = build_mlp(n_in=X_train_p.shape[1], n_out=4)

5. Create callbacks:
   - early_stop = EarlyStopping(monitor="val_accuracy", mode="max", patience=15, restore_best_weights=True)
   - reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5)

6. Train final_mlp with:
   - X_train_p, y_train
   - validation_data=(X_val_p, y_val)
   - epochs=200
   - batch_size=32
   - callbacks=[early_stop, reduce_lr]
   - class_weight=final_cw
   - verbose=0

7. Predict on X_test_p and create:
   - y_test_pred = final_mlp.predict(X_test_p, verbose=0).argmax(axis=1)

8. Print:
   - predicted class counts using np.bincount(y_test_pred, minlength=4)
   - true class counts using np.bincount(y_test, minlength=4)

9. Add a subsection for saving artifacts with this exact section style:

# ---------------------------
# 6.1 ) Extra of saving
# ---------------------------

10. In the saving section:
   - import joblib
   - import os
   - create artifacts directory with os.makedirs("artifacts", exist_ok=True)
   - save fitted preprocessor to artifacts/preprocessor.joblib
   - save class_names to artifacts/class_names.joblib
   - save model weights to artifacts/final_mlp.weights.h5
   - print a short confirmation message

Style requirements:
- Keep formatting similar to a clean notebook
- Include short comments
- Return only Python code
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def generate_test_evaluation_code_agent(model: str = "openai:o4-mini") -> str:
    """
    Generates Python code for held-out test evaluation
    of the final trained multiclass model.
    """
    user_prompt = """
You are a Python machine learning coding assistant.

Write ONLY valid Python code.
Do not include explanations.
Do not include markdown fences.

Assume these variables already exist:
- final_mlp
- X_test_p
- y_test
- class_names
- accuracy_score
- f1_score
- classification_report
- confusion_matrix
- ConfusionMatrixDisplay
- plt

Task:
Write code for this notebook section:

# ----------------------------
# 7) Held-out test evaluation
# ----------------------------
# Goal: Evaluate once on TEST (data the model never saw during training or early stopping)

Requirements:
1. Predict test classes:
   - y_test_pred = final_mlp.predict(X_test_p, verbose=0).argmax(axis=1)

2. Compute:
   - test_acc = accuracy_score(y_test, y_test_pred)
   - test_f1 = f1_score(y_test, y_test_pred, average="weighted")

3. Print:
   - a header line for final held-out test results
   - test accuracy formatted to 3 decimals
   - test weighted F1 formatted to 3 decimals
   - classification_report(y_test, y_test_pred, target_names=class_names, digits=3)

4. Build confusion matrix:
   - cm = confusion_matrix(y_test, y_test_pred)
   - disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
   - disp.plot(values_format="d", xticks_rotation=45)
   - plt.title("Confusion Matrix (Test)")
   - plt.show()

Style requirements:
- Keep formatting similar to a clean notebook
- Include short comments
- Return only Python code
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content.strip()

In [ ]:
def run_generated_pipeline(df):

    from sklearn.pipeline import Pipeline
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.compose import ColumnTransformer
    from sklearn.utils.class_weight import compute_class_weight
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        precision_score,
        recall_score,
        classification_report,
        confusion_matrix,
        ConfusionMatrixDisplay,
    )
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


    import numpy as np
    import pandas as pd
    import tensorflow as tf
    import matplotlib.pyplot as plt
    import scipy
    
    scope = {
    "df": df,    
    "np": np,
    "pd": pd,
    "tf": tf,    
    "plt": plt,   
    "scipy": scipy,
    "Pipeline": Pipeline,
    "SimpleImputer": SimpleImputer,
    "StandardScaler": StandardScaler,
    "OneHotEncoder": OneHotEncoder,
    "ColumnTransformer": ColumnTransformer,
    "compute_class_weight": compute_class_weight,
    "GroupKFold": GroupKFold,
    "accuracy_score": accuracy_score,
    "f1_score": f1_score,
    "precision_score": precision_score,
    "recall_score": recall_score,
    "classification_report": classification_report,
    "confusion_matrix": confusion_matrix,
    "ConfusionMatrixDisplay": ConfusionMatrixDisplay,
    "EarlyStopping": EarlyStopping,
    "ReduceLROnPlateau": ReduceLROnPlateau,
    }
        



    target_task = """
Build a multiclass target from these one-hot columns:
- study_group_healthy
- study_group_pre_diabetes_lifestyle_controlled
- study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled
- study_group_insulin_dependent

Requirements:
- Cast target columns to int
- Check row-wise sum across target columns
- Print how many rows are invalid where sum != 1
- Create a class_names list:
  ["healthy", "prediabetes", "oral_medicated", "insulin_dependent"]
- Print dataframe shape and column list before splitting
"""

    
    exec(generate_dataframe_code_agent(target_task), scope)
    exec(generate_target_code_agent(), scope)

    print("target_cols:", scope.get("target_cols"))
    print("len(target_cols):", len(scope.get("target_cols", [])))
    print("class_names:", scope.get("class_names"))
    print("matching cols in df:", [c for c in scope.get("target_cols", []) if c in df.columns])
    print("missing cols:", [c for c in scope.get("target_cols", []) if c not in df.columns])

    
    exec(splitting_code_agent(), scope)
    exec(generate_preprocessor_code_agent(), scope)

    ###############
    mlp_code= generate_best_mlp_agent(task = "Find the best MLP for this multiclass tabular classification task")
    exec(mlp_code, scope)

    print(type(scope["X_train"]))
    print(type(scope["y_train"]))

    #########################################
    exec(generate_groupkfold_code_agent(), scope)
    cv_summary = scope["cv_summary"]
    cv_feedback = scope["cv_feedback"]
    round_score = scope["round_score"]

    mlp_code = revise_mlp_agent(
    task="Improve the MLP",
    prior_code=mlp_code,
    cv_summary=cv_summary,
    cv_feedback=cv_feedback
    )

    print(scope["best_params"])
    print(type(scope["best_params"]["hidden_units"][0]))
    print(scope["best_params"]["hidden_units"])

    exec(mlp_code, scope)

    ###############################
    feedback_result = run_mlp_feedback_loop(
        scope=scope,
        initial_mlp_code=mlp_code,
        n_rounds=3,
        task="Find the best MLP for this multiclass tabular classification task",
    )

    best_mlp_code = feedback_result["best_code"]

    
    exec(best_mlp_code, scope)

    build_mlp = scope["build_mlp"]
    best_params = scope["best_params"]

    print("Best CV summary:")
    print(feedback_result["best_cv_summary"])

    ##########################
    exec(generate_final_training_code_agent(), scope)
    exec(generate_test_evaluation_code_agent(), scope)

    return scope

In [ ]:
scope = run_generated_pipeline(df)

In [ ]:
import shap
import numpy as np
import pandas as pd

preprocessor = scope["preprocessor"]
mlp_model = scope.get("mlp") or scope.get("model")
X_train = scope["X_train"]
X_test = scope["X_test"]
class_names = scope["class_names"]


X_train_proc = preprocessor.transform(X_train)
X_test_proc = preprocessor.transform(X_test)


if hasattr(X_train_proc, "toarray"):
    X_train_proc = X_train_proc.toarray()
if hasattr(X_test_proc, "toarray"):
    X_test_proc = X_test_proc.toarray()


feature_names = preprocessor.get_feature_names_out()


background_idx = np.random.choice(
    X_train_proc.shape[0],
    min(100, X_train_proc.shape[0]),
    replace=False
)
background = X_train_proc[background_idx]


X_explain = X_test_proc[:50]


explainer = shap.KernelExplainer(mlp_model.predict, background)
shap_values = explainer.shap_values(X_explain)

print("type(shap_values):", type(shap_values))
print("shap_values shape:", np.shape(shap_values))
print("X_explain shape:", X_explain.shape)
print("len(feature_names):", len(feature_names))


if isinstance(shap_values, list):
    for i, class_name in enumerate(class_names):
        print(f"SHAP summary for class: {class_name}")
        shap.summary_plot(
            shap_values[i],
            X_explain,
            feature_names=feature_names,
            rng=0
        )


elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    for i, class_name in enumerate(class_names):
        print(f"SHAP summary for class: {class_name}")
        shap.summary_plot(
            shap_values[:, :, i],
            X_explain,
            feature_names=feature_names,
            rng=0
        )

elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 2:
    print(f"SHAP summary plot")
    shap.summary_plot(
        shap_values,
        X_explain,
        feature_names=feature_names,
        rng=0
    )

else:
    print("Unexpected SHAP output format:", type(shap_values), np.shape(shap_values))

In [ ]:

top_features.index = top_features.index.str.replace("num__", "", regex=False)


top_features = top_features.rename(columns={
    "study_group_healthy": "healthy",
    "study_group_insulin_dependent": "insulin dependent",
    "study_group_oral_medication_and_or_non_insulin_injectable_medication_controlled": "medication controlled",
    "study_group_pre_diabetes_lifestyle_controlled": "pre-diabetic"
})


ax = top_features.plot(
    kind="barh",
    stacked=True,
    figsize=(8, 7),   
    width=0.8
)

ax.set_xlabel("mean(|SHAP value|)", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
ax.set_title("Top SHAP Features by Class", fontsize=12)

ax.tick_params(axis="both", labelsize=7)


ax.legend(
    title="Class",
    fontsize=12,
    title_fontsize=12,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.22),
    ncol=2,
    frameon=False
)

plt.tight_layout()
plt.savefig("shap_ieee.png", dpi=900, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

----------------------------


clean_class_names = [
    "healthy",
    "prediabetes",
    "oral_medicated",
    "insulin_dependent"
]


original_names = list(class_names)

class_map = {
    "healthy": None,
    "prediabetes": None,
    "oral_medicated": None,
    "insulin_dependent": None
}

for i, name in enumerate(original_names):
    low = str(name).lower()
    if "healthy" in low:
        class_map["healthy"] = i
    elif "pre" in low:
        class_map["prediabetes"] = i
    elif "oral" in low or "medication" in low:
        class_map["oral_medicated"] = i
    elif "insulin" in low:
        class_map["insulin_dependent"] = i

ordered_indices = [class_map[c] for c in clean_class_names if class_map[c] is not None]
ordered_names = [c for c in clean_class_names if class_map[c] is not None]
ordered_shap = [shap_by_class[i] for i in ordered_indices]


mean_abs_by_class = np.column_stack([
    np.abs(sv).mean(axis=0) for sv in ordered_shap
])

# total importance across all classes
total_importance = mean_abs_by_class.sum(axis=1)

top_n = 10
top_idx = np.argsort(total_importance)[::-1][:top_n]


plot_values = mean_abs_by_class[top_idx]
plot_features = [feature_names[i] for i in top_idx]

plot_features = [
    f.replace("num__", "")
     .replace("cat__", "")
     .replace("study_group_", "")
     .replace("_", " ")
    for f in plot_features
]


plot_features = [
    f.replace("Elevated A1C Levels (Elevated Blood Sugar)", "Elevated A1C Levels (Elevated Blood Suga")
     .replace("Diabetic Retinopathy (In One Or Both Eyes)", "Dry Eye (In One Or Both Eyes)")
    for f in plot_features
]


plot_values = plot_values[::-1]
plot_features = plot_features[::-1]


plt.style.use("ggplot")
fig, ax = plt.subplots(figsize=(7.2, 4.2))  # compact like second image

left = np.zeros(len(plot_features))

for class_idx, cname in enumerate(ordered_names):
    vals = plot_values[:, class_idx]
    ax.barh(plot_features, vals, left=left, label=cname)
    left += vals

ax.set_title("Top SHAP Features by Class (OpenAI)", fontsize=14)
ax.set_xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=11)
ax.set_ylabel("")

ax.tick_params(axis="x", labelsize=9)
ax.tick_params(axis="y", labelsize=10)


ax.legend(
    title="Class",
    bbox_to_anchor=(1.02, 0.5),
    loc="center left",
    frameon=True,
    fontsize=9,
    title_fontsize=10
)

plt.tight_layout()
plt.show()